In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib scikit-learn

# LSTM Price Prediction Testing
Testing LSTM model on S&P 500 daily data

**Workflow:**
1. Load 2023-2024 data for training
2. Train LSTM model once
3. Test on each month of 2024-2025
4. Visualize predictions vs actual prices

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import your modules
from Data_Handling import get_data_auto
from LSTM_Price_Predictor import LSTMPricePredictor

ModuleNotFoundError: No module named 'pandas'

## 1. Configuration & Data Loading

In [ ]:
# Experiment parameters
SYMBOL = "SPY"  # S&P 500 ETF
TIMEFRAME_UNIT = "Day"
MULTIPLIER = 1

# Training period
TRAIN_START = "2023-01-01"
TRAIN_END = "2024-12-31"

# Testing period
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Model parameters
LOOKBACK_WINDOW = 100
PREDICTION_HORIZON = 10
LSTM_UNITS = 50
NUM_LAYERS = 2
DROPOUT = 0.2
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001
PREDICTION_MODE = 'recursive'  # or 'direct'

# Model persistence
MODEL_NAME = f"lstm_{SYMBOL}_{TRAIN_START}_{TRAIN_END}"
RETRAIN = False  # Set to True to force retraining

In [ ]:
# Load training data
print(f"Loading training data: {TRAIN_START} to {TRAIN_END}")
train_data = get_data_auto(SYMBOL, start=TRAIN_START, end=TRAIN_END, 
                           timeframe_unit=TIMEFRAME_UNIT, multiplier=MULTIPLIER)

print(f"\nTraining data shape: {train_data.shape}")
print(f"Date range: {train_data.index[0]} to {train_data.index[-1]}")
print(f"Number of trading days: {len(train_data)}")
train_data.head()

In [ ]:
# Load testing data
print(f"Loading testing data: {TEST_START} to {TEST_END}")
test_data = get_data_auto(SYMBOL, start=TEST_START, end=TEST_END,
                          timeframe_unit=TIMEFRAME_UNIT, multiplier=MULTIPLIER)

print(f"\nTesting data shape: {test_data.shape}")
print(f"Date range: {test_data.index[0]} to {test_data.index[-1]}")
print(f"Number of trading days: {len(test_data)}")
test_data.head()

## 2. Model Training (or Loading)

In [ ]:
# Initialize model
predictor = LSTMPricePredictor(
    lookback_window=LOOKBACK_WINDOW,
    prediction_horizon=PREDICTION_HORIZON,
    lstm_units=LSTM_UNITS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    prediction_mode=PREDICTION_MODE,
    verbose=1
)

print(f"Model configuration:")
print(f"  Lookback window: {LOOKBACK_WINDOW} candles")
print(f"  Prediction horizon: {PREDICTION_HORIZON} candles")
print(f"  LSTM units: {LSTM_UNITS}")
print(f"  Layers: {NUM_LAYERS}")
print(f"  Prediction mode: {PREDICTION_MODE}")

In [ ]:
# Train or load model
import os

model_path = os.path.join('models', f"{MODEL_NAME}.keras")

if os.path.exists(model_path) and not RETRAIN:
    print(f"\n{'='*60}")
    print("Loading existing model...")
    print(f"{'='*60}")
    predictor.load_model(MODEL_NAME)
else:
    print(f"\n{'='*60}")
    print("Training new model...")
    print(f"{'='*60}")
    predictor.fit(train_data, validation_split=0.2, early_stopping_patience=10)
    
    # Save the trained model
    print(f"\nSaving model as '{MODEL_NAME}'...")
    predictor.save_model(MODEL_NAME)

print("\nModel ready for predictions!")

## 3. View Training History

In [ ]:
# Plot training history (if we just trained)
if predictor.training_history is not None:
    history_df = predictor.get_training_history()
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    ax1 = axes[0]
    ax1.plot(history_df['epoch'], history_df['loss'], label='Training Loss', linewidth=2)
    ax1.plot(history_df['epoch'], history_df['val_loss'], label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss (MSE)')
    ax1.set_title('Training History: Loss')
    ax1.legend()
    ax1.grid(True)
    
    # MAE
    ax2 = axes[1]
    ax2.plot(history_df['epoch'], history_df['mae'], label='Training MAE', linewidth=2)
    ax2.plot(history_df['epoch'], history_df['val_mae'], label='Validation MAE', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('MAE')
    ax2.set_title('Training History: MAE')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("Model was loaded from disk, no training history available.")

## 4. Test on Full Test Period

In [ ]:
# Generate predictions on test data
print("Generating predictions on test data...")
predictions = predictor.predict(test_data)

print(f"\nPredictions generated for {len(predictions)} time points")
print(f"Columns: {predictions.columns.tolist()}")
predictions.head(10)

In [ ]:
# Evaluate performance
print("Evaluating model performance...")
metrics = predictor.evaluate(test_data)
print("\nPerformance Metrics by Prediction Step:")
print(metrics)

## 5. Visualize Predictions

In [ ]:
# Plot: Actual vs Predicted (1-step ahead)
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(predictions.index, predictions['actual_price'], 
        label='Actual Price', linewidth=2, color='black', alpha=0.8)
ax.plot(predictions.index, predictions['pred_price_1'], 
        label='Predicted (1-step)', linewidth=2, color='blue', alpha=0.6)

ax.set_title(f'{SYMBOL} Price Prediction - 1 Step Ahead', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Multiple prediction horizons
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(predictions.index, predictions['actual_price'], 
        label='Actual Price', linewidth=2.5, color='black', alpha=0.9)

# Plot different prediction steps with varying opacity
steps_to_plot = [1, 3, 5, 10] if PREDICTION_HORIZON >= 10 else list(range(1, PREDICTION_HORIZON + 1))
colors = ['blue', 'green', 'orange', 'red']

for i, step in enumerate(steps_to_plot):
    if step <= PREDICTION_HORIZON:
        ax.plot(predictions.index, predictions[f'pred_price_{step}'],
               label=f'Predicted ({step}-step)', linewidth=1.5, 
               color=colors[i % len(colors)], alpha=0.6)

ax.set_title(f'{SYMBOL} Price Prediction - Multiple Horizons', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Prediction errors over time
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Absolute errors
ax1 = axes[0]
for step in [1, 5, 10]:
    if step <= PREDICTION_HORIZON:
        errors = np.abs(predictions['actual_price'] - predictions[f'pred_price_{step}'])
        ax1.plot(predictions.index, errors, label=f'{step}-step error', alpha=0.7)

ax1.set_title('Absolute Prediction Errors', fontsize=12, fontweight='bold')
ax1.set_ylabel('Absolute Error ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Percentage errors
ax2 = axes[1]
for step in [1, 5, 10]:
    if step <= PREDICTION_HORIZON:
        pct_errors = (predictions[f'pred_price_{step}'] - predictions['actual_price']) / predictions['actual_price'] * 100
        ax2.plot(predictions.index, pct_errors, label=f'{step}-step error', alpha=0.7)

ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_title('Percentage Prediction Errors', fontsize=12, fontweight='bold')
ax2.set_ylabel('Percentage Error (%)')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Monthly Performance Analysis

In [ ]:
# Analyze performance by month
predictions['year_month'] = predictions.index.to_period('M')

monthly_metrics = []

for period in predictions['year_month'].unique():
    month_data = predictions[predictions['year_month'] == period]
    
    if len(month_data) > 0:
        # Calculate metrics for 1-step prediction
        actual = month_data['actual_price'].values
        pred = month_data['pred_price_1'].values
        
        mae = np.mean(np.abs(actual - pred))
        rmse = np.sqrt(np.mean((actual - pred) ** 2))
        mape = np.mean(np.abs((actual - pred) / actual)) * 100
        
        monthly_metrics.append({
            'period': str(period),
            'n_days': len(month_data),
            'mae': mae,
            'rmse': rmse,
            'mape': mape
        })

monthly_df = pd.DataFrame(monthly_metrics)
print("\nMonthly Performance (1-step predictions):")
print(monthly_df.to_string(index=False))

In [ ]:
# Visualize monthly performance
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# MAE by month
ax1 = axes[0]
ax1.bar(range(len(monthly_df)), monthly_df['mae'], alpha=0.7, edgecolor='black')
ax1.set_xticks(range(len(monthly_df)))
ax1.set_xticklabels(monthly_df['period'], rotation=45)
ax1.set_title('Mean Absolute Error by Month', fontweight='bold')
ax1.set_ylabel('MAE ($)')
ax1.grid(True, axis='y', alpha=0.3)

# RMSE by month
ax2 = axes[1]
ax2.bar(range(len(monthly_df)), monthly_df['rmse'], alpha=0.7, edgecolor='black', color='orange')
ax2.set_xticks(range(len(monthly_df)))
ax2.set_xticklabels(monthly_df['period'], rotation=45)
ax2.set_title('Root Mean Squared Error by Month', fontweight='bold')
ax2.set_ylabel('RMSE ($)')
ax2.grid(True, axis='y', alpha=0.3)

# MAPE by month
ax3 = axes[2]
ax3.bar(range(len(monthly_df)), monthly_df['mape'], alpha=0.7, edgecolor='black', color='green')
ax3.set_xticks(range(len(monthly_df)))
ax3.set_xticklabels(monthly_df['period'], rotation=45)
ax3.set_title('Mean Absolute Percentage Error by Month', fontweight='bold')
ax3.set_ylabel('MAPE (%)')
ax3.set_xlabel('Month')
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Zoom into Specific Months

In [ ]:
# Select a few months to visualize in detail
months_to_plot = predictions['year_month'].unique()[:4]  # First 4 months

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, period in enumerate(months_to_plot):
    if i >= len(axes):
        break
    
    month_data = predictions[predictions['year_month'] == period]
    
    ax = axes[i]
    ax.plot(month_data.index, month_data['actual_price'], 
           label='Actual', linewidth=2, color='black', marker='o', markersize=3)
    ax.plot(month_data.index, month_data['pred_price_1'], 
           label='Predicted (1-step)', linewidth=2, color='blue', marker='s', markersize=3, alpha=0.7)
    
    ax.set_title(f'{period}', fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle(f'{SYMBOL} Monthly Predictions', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
# Overall summary
print("="*60)
print("OVERALL PERFORMANCE SUMMARY")
print("="*60)
print(f"\nSymbol: {SYMBOL}")
print(f"Training Period: {TRAIN_START} to {TRAIN_END}")
print(f"Testing Period: {TEST_START} to {TEST_END}")
print(f"\nModel Configuration:")
print(f"  Lookback Window: {LOOKBACK_WINDOW} candles")
print(f"  Prediction Horizon: {PREDICTION_HORIZON} candles")
print(f"  LSTM Units: {LSTM_UNITS}")
print(f"  Layers: {NUM_LAYERS}")
print(f"  Prediction Mode: {PREDICTION_MODE}")
print(f"\nTest Period Statistics:")
print(f"  Total trading days: {len(predictions)}")
print(f"  Months evaluated: {len(monthly_df)}")
print(f"\nAverage Monthly Performance (1-step):")
print(f"  MAE: ${monthly_df['mae'].mean():.2f}")
print(f"  RMSE: ${monthly_df['rmse'].mean():.2f}")
print(f"  MAPE: {monthly_df['mape'].mean():.2f}%")
print(f"\nBest Month: {monthly_df.loc[monthly_df['mae'].idxmin(), 'period']} (MAE: ${monthly_df['mae'].min():.2f})")
print(f"Worst Month: {monthly_df.loc[monthly_df['mae'].idxmax(), 'period']} (MAE: ${monthly_df['mae'].max():.2f})")
print("="*60)

## Next Steps

**Experiment Ideas:**
1. Try `prediction_mode='direct'` and compare results
2. Adjust `LOOKBACK_WINDOW` (50, 150, 200)
3. Change `PREDICTION_HORIZON` to test different forecasting lengths
4. Increase `LSTM_UNITS` or `NUM_LAYERS` for more capacity
5. Test on different symbols (NVDA, AAPL, etc.)
6. Implement walk-forward retraining (Option B)